In [7]:
from google.colab import drive
drive.mount('/content/drive')
base_dir = "/content/drive/MyDrive/huggingface-rag"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install qdrant-client sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer

output_path = f"{base_dir}/ft-jina-transformers-v1"
model = SentenceTransformer(output_path, trust_remote_code=True)
embedding_size = model.get_sentence_embedding_dimension()
print(f"Model embedding size: {embedding_size}")

Model embedding size: 768


In [12]:
import os
from qdrant_client import QdrantClient
from qdrant_client.http import models

qdrant_path = f"{base_dir}/qdrant_db"

lock_file = os.path.join(qdrant_path, ".lock")
if os.path.exists(lock_file):
  try:
    os.remove(lock_file)
    print(f"Removed stale lock file: {lock_file}")
  except Exception as e:
    print(f"Warning: Could not remove lock file: {e}")

client = QdrantClient(path=qdrant_path)
collection_name = 'huggingface_transformers_docs'

client.delete_collection(collection_name)

client.create_collection(
  collection_name=collection_name,
  vectors_config=models.VectorParams(
    size=embedding_size,
    distance=models.Distance.COSINE
  ),
)
print(f"Collection '{collection_name}' created.")

Removed stale lock file: /content/drive/MyDrive/huggingface-rag/qdrant_db/.lock
Collection 'huggingface_transformers_docs' created.


In [13]:
from tqdm import tqdm
import uuid
import json

batch_size = 64
chunked_path = f"{base_dir}/chunks.jsonl"

with open(chunked_path, 'r', encoding='utf-8') as f_in:
  total_docs = sum(1 for line in f_in if line.strip())

with open(chunked_path, 'r', encoding='utf-8') as f_in:
  batch_docs = []
  for line in tqdm(f_in, desc="Building document index", total=total_docs):
    line = line.strip()
    if not line:
      continue

    doc = json.loads(line)
    batch_docs.append(doc)

    if len(batch_docs) >= batch_size:
      batch_texts = [doc['text'] for doc in batch_docs]
      embeddings = model.encode(batch_texts, convert_to_tensor=False)

      points = []
      for idx, embedding in enumerate(embeddings):
        doc = batch_docs[idx]
        payload = doc
        points.append(models.PointStruct(
          id=str(uuid.uuid4()),
          vector=embedding.tolist(),
          payload=payload
        ))

      client.upsert(
        collection_name=collection_name,
        points=points
      )

      batch_docs = []

if batch_docs:
  batch_texts = [doc["text"] for doc in batch_docs]
  embeddings = model.encode(batch_texts, convert_to_tensor=False)

  points = []
  for idx, vector in enumerate(embeddings):
    doc = batch_docs[idx]
    payload = doc

    points.append(models.PointStruct(
      id=str(uuid.uuid4()),
      vector=vector.tolist(),
      payload=payload
    ))

  client.upsert(
    collection_name=collection_name,
    points=points
  )

print("Index building complete")

Building document index: 100%|██████████| 6473/6473 [03:47<00:00, 28.39it/s]


Index building complete


In [14]:
import qdrant_client
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Distance, VectorParams
import json
import os
import time

qdrant_path = f"{base_dir}/qdrant_db"
collection_name = 'huggingface_transformers_docs'

print(f"Connecting to Qdrant at: {qdrant_path}")

if 'client' in globals():
  try:
    if hasattr(client, 'close'):
      client.close()
    print("Closed existing Qdrant client.")
  except Exception as e:
    print(f"Warning closing client: {e}")

# Force clean up any remaining .lock files
lock_file = os.path.join(qdrant_path, ".lock")
if os.path.exists(lock_file):
  try:
    os.remove(lock_file)
    print(f"Removed stale lock file: {lock_file}")
  except Exception as e:
    print(f"Warning: Could not remove lock file: {e}")

# Initialize client
try:
  client = QdrantClient(path=qdrant_path)
  print(f"Client initialized: {type(client)}")
  if not client.collection_exists(collection_name):
    raise ValueError(f"Collection '{collection_name}' does not exist! Please run ingestion first.")
except Exception as e:
  print(f"Error initializing client: {e}")
  raise e


# ==========================================
# 1. Basic Semantic Search Test
# ==========================================
def test_basic_search(query, top_k=5):
  """Test basic semantic search"""
  print(f"\n🔍 Query: {query}")
  print("=" * 80)

  query_vector = model.encode(query).tolist()

  try:
    # Compatibility handling: try search first, fallback to query_points
    if hasattr(client, 'search'):
      results = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=top_k
      )
    else:
      print("⚠️ 'search' method missing. Trying 'query_points'...")
      results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True
      ).points

    for idx, hit in enumerate(results, 1):
      # Get metadata
      meta = hit.payload.get('metadata', {})
      source = meta.get('source', 'N/A')
      headers = ' > '.join(meta.get('headers', []))

      print(f"\n[Result {idx}] Score: {hit.score:.4f}")
      print(f"Source:  {source}") # Verify if this shows as relative path
      print(f"Headers: {headers}")
      print(f"Text:    {hit.payload['text'][:200].replace('\n', ' ')}...") # Remove newlines for easier viewing

    return results

  except Exception as e:
    print(f"❌ Error executing search: {e}")
    return []

print("\n" + "="*80)
print("Basic Semantic Search Test")
print("="*80)

test_basic_search("How to choose GPU for deep learning?")
test_basic_search("What is text generation in transformers?")
test_basic_search("How to handle padding in tokenization?")


# ==========================================
# 2. Metadata Filtering Test (Key modification section)
# ==========================================
def test_metadata_filtering(query, source_filter=None, header_filter=None, top_k=3):
  """Test search with metadata filtering"""
  print(f"\n🔍 Query: {query}")

  filter_conditions = []

  if source_filter:
    print(f"📁 Filter by source: {source_filter}")
    filter_conditions.append(
      FieldCondition(
        key="metadata.source",
        match=MatchValue(value=source_filter)
      )
    )

  if header_filter:
    print(f"📑 Filter by header: {header_filter}")
    filter_conditions.append(
      FieldCondition(
        key="metadata.headers",
        match=MatchValue(value=header_filter)
      )
    )

  print("=" * 80)

  query_vector = model.encode(query).tolist()
  query_filter = Filter(must=filter_conditions) if filter_conditions else None

  try:
    if hasattr(client, 'search'):
      results = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        query_filter=query_filter,
        limit=top_k
      )
    else:
      results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=query_filter,
        limit=top_k,
        with_payload=True
      ).points

    print(f"Found {len(results)} results")
    for idx, hit in enumerate(results, 1):
      meta = hit.payload.get('metadata', {})
      print(f"\n[Result {idx}] Score: {hit.score:.4f}")
      print(f"Source: {meta.get('source', 'N/A')}") # Confirm if filtering works
      print(f"Text: {hit.payload['text'][:150]}...")

    return results
  except Exception as e:
    print(f"Skipping metadata test due to error: {e}")
    return []

print("\n" + "="*80)
print("Metadata Filtering Test")
print("="*80)

test_metadata_filtering(
  "GPU power requirements",
  source_filter="perf_hardware.md"
)

# Test subdirectory filtering (assuming bert.md is under model_doc, you can modify this test based on actual situation)
# test_metadata_filtering(
#   "BERT configuration",
#   source_filter="model_doc/bert.md"
# )

test_metadata_filtering(
  "padding and truncation",
  header_filter="Padding and truncation"
)


# ==========================================
# 3. Diversity Test
# ==========================================
def test_diverse_queries():
  queries = [
    "What are NVLink benefits?",
    "GPU cooling temperature requirements",
    "What is LOMO optimizer?",
    "How to resample audio data?",
    "Decoding strategies for creative text",
    "CUDA out of memory solutions",
  ]

  print("\n" + "="*80)
  print("Diverse Query Test")
  print("="*80)

  for query in queries:
    query_vector = model.encode(query).tolist()
    try:
      if hasattr(client, 'search'):
        results = client.search(collection_name=collection_name, query_vector=query_vector, limit=1)
      else:
        results = client.query_points(collection_name=collection_name, query=query_vector, limit=1, with_payload=True).points

      if results:
        top_score = results[0].score
        # Now directly get source, no need to split since it's already relative path
        top_source = results[0].payload.get('metadata', {}).get('source', 'N/A')
        print(f"✓ {query:<40} | Score: {top_score:.4f} | File: {top_source}")
      else:
        print(f"✗ {query:<40} | No results found")

    except Exception as e:
      print(f"✗ {query}: Error - {e}")

test_diverse_queries()


# ==========================================
# 4. Context Comparison Test (Verify if 'Context: ...' prefix is needed)
# ==========================================
def compare_with_without_context(query, top_k=3):
  print(f"\n" + "="*80)
  print(f"Comparison Test: {query}")
  print("="*80)

  # 1. Simple query
  print("\n👉 Simple query:")
  simple_vec = model.encode(query).tolist()

  # Use query_points (better compatibility)
  simple_hits = client.query_points(
    collection_name=collection_name,
    query=simple_vec,
    limit=top_k,
    with_payload=True
  ).points if not hasattr(client, 'search') else client.search(
    collection_name=collection_name, query_vector=simple_vec, limit=top_k
  )

  for idx, hit in enumerate(simple_hits, 1):
    print(f"  [{idx}] Score: {hit.score:.4f} | {hit.payload['text'][:80].replace('\n', ' ')}...")

  # 2. With context prefix
  context_q = f"In the context of Hugging Face Transformers library: {query}"
  print(f"\n👉 Contextual query ('{context_q}'):")
  context_vec = model.encode(context_q).tolist()

  context_hits = client.query_points(
    collection_name=collection_name,
    query=context_vec,
    limit=top_k,
    with_payload=True
  ).points if not hasattr(client, 'search') else client.search(
    collection_name=collection_name, query_vector=context_vec, limit=top_k
  )

  for idx, hit in enumerate(context_hits, 1):
    print(f"  [{idx}] Score: {hit.score:.4f} | {hit.payload['text'][:80].replace('\n', ' ')}...")

compare_with_without_context("padding")


# ==========================================
# 5. Statistical Analysis (Verify deduplication and path structure)
# ==========================================
def analyze_collection():
  print("\n" + "="*80)
  print("Collection Statistical Analysis")
  print("="*80)

  try:
    # Get collection information
    collection_info = client.get_collection(collection_name)
    print(f"\n📊 Total vectors: {collection_info.points_count}")

    # Random sampling to view path distribution
    sample_size = 100
    scroll_result = client.scroll(
      collection_name=collection_name,
      limit=sample_size
    )

    sources = {}
    for point in scroll_result[0]:
      # Directly count original source to view directory distribution
      full_source_path = point.payload.get('metadata', {}).get('source', 'Unknown')
      sources[full_source_path] = sources.get(full_source_path, 0) + 1

    print(f"\nTop Source Files (from {sample_size} random samples):")
    # Sort by frequency
    sorted_sources = sorted(sources.items(), key=lambda x: x[1], reverse=True)
    for source, count in sorted_sources[:15]:
      print(f"  {count} chunks | {source}")

  except Exception as e:
    print(f"Error in analysis: {e}")

analyze_collection()


# ==========================================
# 6. Threshold Test
# ==========================================
def test_score_threshold(query, threshold=0.5):
  print(f"\n" + "="*80)
  print(f"Threshold Test: '{query}' (threshold={threshold})")
  print("="*80)

  query_vector = model.encode(query).tolist()

  try:
    if hasattr(client, 'search'):
      results = client.search(collection_name=collection_name, query_vector=query_vector, limit=5, score_threshold=threshold)
    else:
      results = client.query_points(collection_name=collection_name, query=query_vector, limit=5, score_threshold=threshold, with_payload=True).points

    if not results:
      print("No results found above threshold.")
    else:
      for idx, hit in enumerate(results, 1):
        print(f"[{idx}] Score: {hit.score:.4f} | {hit.payload['text'][:100].replace('\n', ' ')}...")

  except Exception as e:
    print(f"Error: {e}")

test_score_threshold("GPU memory optimization", threshold=0.45)
test_score_threshold("how to cook pasta", threshold=0.45)

Connecting to Qdrant at: /content/drive/MyDrive/huggingface-rag/qdrant_db
Closed existing Qdrant client.
Removed stale lock file: /content/drive/MyDrive/huggingface-rag/qdrant_db/.lock
Client initialized: <class 'qdrant_client.qdrant_client.QdrantClient'>

Basic Semantic Search Test

🔍 Query: How to choose GPU for deep learning?
⚠️ 'search' method missing. Trying 'query_points'...

[Result 1] Score: 0.6295
Source:  perf_hardware.md
Headers: Build your own machine
Text:    Context: Build your own machine  # Build your own machine  One of the most important considerations when building a machine for deep learning is the GPU choice. GPUs are the standard workhorse for dee...

[Result 2] Score: 0.5982
Source:  perf_train_gpu_one.md
Headers: GPU
Text:    Context: GPU  # GPU  GPUs are commonly used to train deep learning models due to their high memory bandwidth and parallel processing capabilities. Depending on your GPU and model size, it is possible ...

[Result 3] Score: 0.5356
Source:  p